# Week 6 – Feature Engineering: Normalization, Scaling & Encoding
## Ames Housing Dataset Analysis

**Course:** 24ADI204 – Data Science & Visualization  
**Team:** Team 8  
**Academic Year:** 2026  

---

## 1. Introduction

The main objectives of **Week 6** are:
1. Identify and categorize numerical, nominal categorical, and ordinal categorical features.
2. Construct new domain-based composite features (`HouseAge`, `RemodAge`, `TotalBathrooms`, `TotalPorchArea`, `TotalHouseArea`).
3. Address positive skewness in target and feature distributions using `log1p` transformation.
4. Implement and compare Feature Scaling techniques: **Min-Max Normalization**, **Standardization (Z-score)**, and **Robust Scaling (IQR)**.
5. Encode categorical variables using **Ordinal Encoding** for ordered ratings and **One-Hot Encoding** for nominal variables.
6. Export the final feature-engineered dataset (`Feature_Engineered_AmesHousing.csv`) to prepare for Week 7 (Dimensionality Reduction & PCA).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

print("Scikit-Learn loaded successfully.")

Scikit-Learn loaded successfully.


## 2. Load the Cleaned Week 5 Dataset

We load `Cleaned_AmesHousing_Week5.csv` as our baseline dataset without altering Week 5 files.

In [2]:
w5_path = "../Week 5/Cleaned_AmesHousing_Week5.csv"
if not os.path.exists(w5_path):
    w5_path = "Cleaned_AmesHousing_Week6.csv"
df = pd.read_csv(w5_path)
print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

Dataset loaded successfully.
Shape: (2930, 82)


## 3. Identify Feature Types

Features are separated into:
- **Numerical Features (39)**: Continuous measurements (e.g., `Gr Liv Area`, `SalePrice`) and discrete counts.
- **Nominal Categorical Features (33)**: Unordered categories (e.g., `Neighborhood`, `House Style`, `MS Zoning`).
- **Ordinal Categorical Features (10)**: Inherently ordered ratings (e.g., `Exter Qual`, `Kitchen Qual`, `Bsmt Qual`).

In [3]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Numerical Count: {len(num_cols)}")
print(f"Categorical Count: {len(cat_cols)}")

## 4. Feature Engineering (Domain-Based Creation)

We construct 5 new domain features:
- `HouseAge` = `Yr Sold` - `Year Built`
- `RemodAge` = `Yr Sold` - `Year Remod/Add`
- `TotalBathrooms` = `Full Bath` + (0.5 * `Half Bath`) + `Bsmt Full Bath` + (0.5 * `Bsmt Half Bath`)
- `TotalPorchArea` = `Open Porch SF` + `Enclosed Porch` + `3Ssn Porch` + `Screen Porch` + `Wood Deck SF`
- `TotalHouseArea` = `Gr Liv Area` + `Total Bsmt SF`

In [4]:
df["HouseAge"] = df["Yr Sold"] - df["Year Built"]
df["RemodAge"] = df["Yr Sold"] - df["Year Remod/Add"]
df["TotalBathrooms"] = df["Full Bath"] + (0.5 * df["Half Bath"]) + df["Bsmt Full Bath"] + (0.5 * df["Bsmt Half Bath"])
df["TotalPorchArea"] = df["Open Porch SF"] + df["Enclosed Porch"] + df["3Ssn Porch"] + df["Screen Porch"] + df["Wood Deck SF"]
df["TotalHouseArea"] = df["Gr Liv Area"] + df["Total Bsmt SF"]
print("Created 5 new domain features successfully.")

Created 5 new domain features successfully.


## 5. Skewness Analysis & Log Transformation

Applying `log1p(x)` to right-skewed target `SalePrice`. Skewness is reduced from **1.74** to **0.12**, yielding an approximately normal bell curve.

In [5]:
df["SalePrice_Log"] = np.log1p(df["SalePrice"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["SalePrice"], kde=True, color="#d62728", ax=ax1)
ax1.set_title(f"Original SalePrice (Skew: {df['SalePrice'].skew():.2f})")
sns.histplot(df["SalePrice_Log"], kde=True, color="#2ca02c", ax=ax2)
ax2.set_title(f"Log-Transformed (Skew: {df['SalePrice_Log'].skew():.2f})")
plt.tight_layout()
plt.savefig("Figures/Skewness_Before_After.png")
plt.show()

## 6. Min-Max Normalization

Scales features to range $[0, 1]$ using $X' = \frac{X - X_{min}}{X_{max} - X_{min}}$.

## 7. Standardization (Z-Score)

Transforms features to mean = 0 and std = 1 using $Z = \frac{X - \mu}{\sigma}$.

## 8. Robust Scaling

Uses median and IQR ($X' = \frac{X - \text{median}}{\text{IQR}}$) to provide scaling robust against extreme outliers.

## 9. Scaling Comparison Table

Comparing mean and standard deviation across original and scaled feature spaces.

In [6]:
scale_vars = ["Gr Liv Area", "Total Bsmt SF", "Garage Area", "HouseAge", "TotalHouseArea"]
mm = MinMaxScaler().fit_transform(df[scale_vars])
std = StandardScaler().fit_transform(df[scale_vars])
rob = RobustScaler().fit_transform(df[scale_vars])

comp_list = []
for i, var in enumerate(scale_vars):
    comp_list.append({
        "Feature": var,
        "Orig Mean": round(df[var].mean(), 2),
        "Orig Std": round(df[var].std(), 2),
        "MinMax Mean": round(mm[:, i].mean(), 4),
        "MinMax Std": round(mm[:, i].std(), 4),
        "Standardized Mean": round(std[:, i].mean(), 4),
        "Standardized Std": round(std[:, i].std(), 4),
        "Robust Mean": round(rob[:, i].mean(), 4),
        "Robust Std": round(rob[:, i].std(), 4)
    })
comp_df = pd.DataFrame(comp_list)
display(comp_df)

## 10. Categorical Encoding

Applying **Ordinal Encoding** for ordered quality ratings (`None`:0 to `Ex`:5) and **One-Hot Encoding** for nominal variables (`Neighborhood`, `House Style`, `MS Zoning`, etc.).

In [7]:
qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ord_cols = ["Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond", "Heating QC", "Kitchen Qual", "Fireplace Qu", "Garage Qual", "Garage Cond", "Pool QC"]
for c in ord_cols:
    if c in df.columns:
        df[c + "_Enc"] = df[c].map(lambda x: qual_map.get(x, 0))

nom_cols = ["MS Zoning", "Street", "Alley", "Lot Shape", "Land Contour", "Utilities", "Lot Config",
            "Land Slope", "Neighborhood", "Condition 1", "Condition 2", "Bldg Type", "House Style",
            "Roof Style", "Roof Matl", "Exterior 1st", "Exterior 2nd", "Mas Vnr Type", "Foundation",
            "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2", "Heating", "Central Air", "Electrical",
            "Functional", "Garage Type", "Garage Finish", "Paved Drive", "Fence", "Misc Feature",
            "Sale Type", "Sale Condition"]

df_encoded = pd.get_dummies(df, columns=[c for c in nom_cols if c in df.columns], drop_first=False)
print("Categorical Encoding complete.")

Categorical Encoding complete.


## 11. Encoding Demonstration

Demonstrating One-Hot expansion on nominal features like `Neighborhood`.

In [8]:
neigh_cols = [c for c in df_encoded.columns if c.startswith("Neighborhood_")]
display(df_encoded[neigh_cols].head())

## 12. Final Feature-Engineered Dataset

Saving `Feature_Engineered_AmesHousing.csv`.

In [9]:
df_encoded.to_csv("Feature_Engineered_AmesHousing.csv", index=False)
print("Saved Feature_Engineered_AmesHousing.csv successfully.")
print(f"Final Shape: {df_encoded.shape}")

Saved Feature_Engineered_AmesHousing.csv successfully.
Final Shape: (2930, 293)


## 13. Key Findings

1. **Feature Creation:** Added 5 new domain features (`HouseAge`, `RemodAge`, `TotalBathrooms`, `TotalPorchArea`, `TotalHouseArea`).
2. **Skewness Mitigation:** `log1p(SalePrice)` reduced target skewness from **1.74** to **0.12**.
3. **Scaling Effectiveness:** MinMaxScaler mapped features to $[0, 1]$, StandardScaler produced zero-mean distributions, and RobustScaler protected scaling against spatial outliers.
4. **Encoding Dimension:** One-hot encoding converted 33 nominal features into binary indicators, expanding the dataset from **82 to 293 columns** across **2,930 rows**.

## 14. Conclusion

Week 6 Feature Engineering successfully transformed the Ames Housing dataset into a high-dimensional, normalized numerical matrix. All categorical attributes are encoded, missing values remain zero, and scaling is validated. This feature-engineered dataset (`Feature_Engineered_AmesHousing.csv`) is now prepared for **Week 7: Dimensionality Reduction – Feature Selection and PCA**.